# WW3 on an Unstructured Mesh (experimental)

```{admonition} Experimental
:class: warning

This notebook runs WaveWatch III (WW3) on an **unstructured triangular mesh** inside a
CrocoDash regional CESM case (MOM6 + WW3). It needs an **unstructured-capable CESM**
checkout (GLADE/derecho only for now — see `cesmroot` below) and is **not** part of the
standard CrocoDash release. It is a prototype for the workshop, not a product: expect
rough edges (the implicit solver never fully converges within its iteration cap, the
mesh code has no pole handling, and CICE coupling on an unstructured mesh is untested).
Please try it, try to break it, and report problems (mesh gotchas, build/run failures,
namelist surprises) to the CROCODILE-CESM team.

This notebook deliberately does **not** include any fancy mesh-generation
infrastructure — that lives in external tools. It offers only (A) a simple built-in
mesh made by triangulating the wet MOM6 cells, or (B) a mesh you bring yourself (gmsh
v2 format).
```

This is the [Ocean + Waves](tutorial-ocn-wav.ipynb) tutorial with one addition: after
forcing processing and before the build, a new section replaces WW3's usual structured
(`CURV`) grid — a 1:1 copy of the MOM6 grid — with an unstructured triangular mesh.

A typical workflow of utilizing CrocoDash consists of four main steps:

1. Generate a regional MOM6 domain.
2. Create the CESM case.
3. Prepare forcing data.
4. Build and run the case — after configuring WW3's unstructured mesh (new Section 4
   below), inserted between steps 3 and 4.

## Parameters

Override any of these to test a different domain, run length, mesh, or WW3 numerical
scheme. Defaults give a **short test**: 5 days of forcing, a 2-day run. To lengthen the
run, widen `date_range` and raise `run_length_days` together (forcing must cover the
run length plus a little buffer for restart/output alignment).

In [ ]:
# CESM case name
casename = "bering_unst_ww3.001"

# Forcing window (must be >= run_length_days + ~1 day buffer) and run length
date_range = ["2020-03-01 00:00:00", "2020-03-06 00:00:00"]  # 5 days
run_length_days = 2

# Mesh source: "mom6" builds a mesh from the wet MOM6 T-cells (Section 4a below);
# any other value is treated as a path to a bring-your-own gmsh v2 (.msh) mesh file.
mesh_source = "mom6"

# WW3 unstructured numerical scheme: "implicit" (robust default, PDLIB/Jacobi-Gauss-
# Seidel) or "explicit" (CFL-limited timestep; needs a fine enough mesh to be stable).
scheme = "implicit"

# WW3 grid name (WAV_GRID / GRID%NAME)
wave_grid_name = "bering_unst"

# SECTION 1: Generate a regional MOM6 domain

We begin by defining a regional MOM6 domain using CrocoDash. To do so, we first generate a horizontal grid. We then generate the topography by remapping an existing bathymetric dataset to our horizontal grid. Finally, we define a vertical grid.

## Step 1.1: Horizontal Grid

📖 [Grid documentation](https://crocodile-cesm.github.io/CrocoDash/latest/for_users/1_grids.html)

In [ ]:
from CrocoDash.grid import Grid

grid = Grid(
  resolution = 1/8, # in degrees
  xstart = 177, # min longitude in [0, 360]
  lenx = 27.0, # longitude extent in degrees
  ystart = 55.0, # min latitude in [-90, 90]
  leny = 12.0, # latitude extent in degrees
  name = "bering",
)

## Step 1.2: Topography


In [ ]:
from CrocoDash.topo import Topo

topo = Topo(
    grid = grid,
    min_depth = 9.5, # in meters
)

In [ ]:
from pathlib import Path

bathymetry_path= Path("<GEBCO_LOWRES>")
# The GEBCO bathymetry path on Derecho is: <GEBCO>

if not bathymetry_path.exists():
    raise FileNotFoundError("Bathymetry file not found, please replace with path to bathymetry file")

topo.set_from_dataset(
    bathymetry_path = bathymetry_path,
    longitude_coordinate_name="lon",
    latitude_coordinate_name="lat",
    vertical_coordinate_name="elevation",
    depth_method="xesmf",
    mask_method="dataset",
)

In [ ]:
topo.basintmask.plot(size=4, aspect = grid.nx / grid.ny)

In [ ]:
topo.keep_largest_basin()
topo.basintmask.plot(size=4, aspect = grid.nx / grid.ny)

In [ ]:
## Optionally, uncomment the below lines to further modify the topography interactively
#%matplotlib ipympl
#from CrocoDash.topo_editor import TopoEditor
#TopoEditor(topo)

## Step 1.3: Vertical Grid

📖 [Grid documentation](https://crocodile-cesm.github.io/CrocoDash/latest/for_users/1_grids.html)

In [ ]:
from CrocoDash.vgrid import VGrid

vgrid  = VGrid.hyperbolic(
    nk = 32, # number of vertical levels
    depth = topo.max_depth,
    ratio=50.0 # target ratio of top to bottom layer thicknesses
)

In [ ]:
import matplotlib.pyplot as plt
plt.close()
# Create the plot
for depth in vgrid.zi:
    plt.axhline(y=depth, linestyle='-')  # Horizontal lines

plt.ylim(max(vgrid.zi) + 10, min(vgrid.zi) - 10)  # Invert y-axis so deeper values go down
plt.ylabel("Depth")
plt.title("Vertical Grid")
plt.show()

# SECTION 2: Create the CESM case

After generating the MOM6 domain, the next step is to create a CESM case using CrocoDash. The Case object requires the following inputs:

 - CESM Source Directory: a local path to a compatible CESM source copy — for **unstructured WW3**, this must be an unstructured-capable checkout (see `cesmroot` below), not the standard workshop CESM.
 - Case Name: a unique name for the CESM case.
 - Input Directory: where CrocoDash writes the grid, bathymetry and forcing files.
 - MOM6 Domain Objects: the horizontal grid, topography, and vertical grid from Section 1.
 - Project ID / compset: as usual.

## Step 2.1: Specify case name and directories


In [ ]:
from pathlib import Path

# CESM source root. Standard CrocoDash cesmroot will NOT build unstructured WW3: it needs
# 3 infrastructure patches (ParMETIS linked into WW3's PDLIB build, WW3 buildnml accepting
# ww3_grid.nml as grid input, and WW3_PARMETIS_ROOT set for the target machine) that are not
# yet in the workshop CESM. This default points at a checkout that already has them (see
# cesm_unstr/UNSTR_README.md); check_cesmroot_patches() below verifies this at run time.
cesmroot = "/glade/work/damrhein/cesm_unstr/"

# Place where all your input files go
inputdir = Path("<inputdir>") / casename

# CESM case directory
caseroot = Path("<casedir>") / casename

```{admonition} Unstructured-capable CESM check
:class: note

`check_cesmroot_patches()` inspects `cesmroot` for the 3 infrastructure changes this
notebook depends on and raises a clear error naming whichever is missing, instead of
failing much later at build or submit time.
```

In [ ]:
def check_cesmroot_patches(cesmroot):
    """Raise a clear error unless `cesmroot` has the unstructured-WW3 infrastructure
    patches (see cesm_unstr/UNSTR_README.md): ParMETIS linked into WW3's PDLIB build,
    WW3 buildnml accepting ww3_grid.nml, and WW3_PARMETIS_ROOT set for derecho."""
    cesmroot = Path(cesmroot)
    checks = {
        "WW3 model/src/CMakeLists.txt links parmetis for PDLIB builds": (
            cesmroot / "components/ww3/WW3/model/src/CMakeLists.txt",
            lambda t: "find_library" in t and "parmetis" in t.lower(),
        ),
        "components/ww3/cime_config/buildnml accepts ww3_grid.nml": (
            cesmroot / "components/ww3/cime_config/buildnml",
            lambda t: "ww3_grid.nml" in t,
        ),
        "ccs_config derecho config_machines.xml sets WW3_PARMETIS_ROOT": (
            cesmroot / "ccs_config/machines/derecho/config_machines.xml",
            lambda t: "WW3_PARMETIS_ROOT" in t,
        ),
    }
    missing = [f"  - {label} ({path})" for label, (path, test) in checks.items()
               if not path.exists() or not test(path.read_text())]
    if missing:
        raise RuntimeError(
            f"cesmroot does not look like an unstructured-WW3-capable CESM checkout "
            f"({cesmroot}). Missing/failed checks:\n" + "\n".join(missing) +
            "\nSee cesm_unstr/UNSTR_README.md and point cesmroot at a patched checkout."
        )


check_cesmroot_patches(cesmroot)
print(f"cesmroot OK: {cesmroot}")

## Step 2.2: Create the Case

The `job_queue` and `job_wallclock_time` arguments set the PBS queue and the walltime (`hh:mm:ss`) CESM requests when you submit the run; leave them out to use the machine defaults.

📖 [Case documentation](https://crocodile-cesm.github.io/CrocoDash/latest/for_users/2_case_setup.html)

### Compset quick-reference

| What you want | Change in `compset=` |
|---|---|
| Ocean only (default) | `compset="CR_JRA"` |
| + **Waves (WW3)** | **`compset="CWR_JRA"`** |
| + Sea ice (CICE6) | change `CWR_JRA` → `GWR_JRA` (untested on an unstructured wave mesh — see Known limitations) |

In [ ]:
from CrocoDash.case import Case

case = Case(
    cesmroot = cesmroot,
    caseroot = caseroot,
    inputdir = inputdir,
    ocn_grid = grid,
    ocn_vgrid = vgrid,
    ocn_topo = topo,
    project = '<PROJECT>', # Enter Project ID for PBS Queue
    job_queue = "<JOB_QUEUE>", # PBS queue the run is submitted to
    job_wallclock_time = "<WALLCLOCK_1H>", # hh:mm:ss walltime requested for the run
    override = True,
    machine = "derecho",
    compset = "CWR_JRA"
)

# Section 3: Prepare ocean and wave forcing data

We need to cut out our ocean and waves forcing. `"CESM-WW3-JRA"` is spectra from a global 1-degree CESM-WW3 run forced by the same JRA winds.

In [ ]:
boundaries = ["south", "north", "west"]  # open MOM6 edges; reused below for mesh boundary classification

case.configure_forcings(
    date_range = date_range,
    boundaries=boundaries,
    product_name = "GLORYS",
    function_name="get_glorys_data_from_rda",
    ww3_obc_product_name='CESM-WW3-JRA',
    ww3_obc_function_name="get_cesm_ww3_jra_spectra",
)

##  Step 3.3: Process forcing data


In [ ]:
case.process_forcings()

# Section 4: WW3 on an unstructured mesh

This replaces WW3's usual structured (`CURV`) grid — written by `process_forcings()`
above into `<inputdir>/wav/` — with an **unstructured triangular mesh**, before the
first build. Nothing here touches the ocean (MOM6) side of the case.

## Step 4a: Build (or load) the mesh

- **`mesh_from_mom6`**: nodes at wet T-cell centres; each 2×2 block of neighbouring
  centres becomes 2 triangles (split along the shorter diagonal) if all 4 corners are
  wet, or 1 triangle if exactly 3 are wet. Keeps the largest connected component and
  removes non-manifold ("bow-tie") vertices, iterating to a fixed point.
- **`read_msh`**: loads a bring-your-own gmsh v2 mesh. Requirements: lon/lat in
  degrees; **continuous** longitudes (0–360, not wrapped, for domains crossing the
  dateline); depth **positive**, in metres; **triangles only**; the mesh must not
  contain a pole.

In [ ]:
import numpy as np


def _haversine_km(lon1, lat1, lon2, lat2):
    R = 6371.0088
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp, dl = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dp / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0)))


def _compact(tris, keep_gids, total_size):
    remap = -np.ones(total_size, dtype=np.int64)
    remap[keep_gids] = np.arange(len(keep_gids))
    a, b, c = remap[tris[:, 0]], remap[tris[:, 1]], remap[tris[:, 2]]
    good = (a >= 0) & (b >= 0) & (c >= 0)
    return np.stack([a[good], b[good], c[good]], axis=1)


def _largest_component(node_gids, tris):
    n = len(node_gids)
    parent = np.arange(n)

    def find(x):
        r = x
        while parent[r] != r:
            r = parent[r]
        while parent[x] != r:
            parent[x], x = r, parent[x]
        return r

    for a, b, c in tris:
        for x, y in ((a, b), (b, c)):
            rx, ry = find(x), find(y)
            if rx != ry:
                parent[rx] = ry
    roots = np.array([find(x) for x in range(n)])
    touched = np.zeros(n, dtype=bool)
    touched[tris.reshape(-1)] = True
    vals, counts = np.unique(roots[touched], return_counts=True)
    keep = (roots == vals[np.argmax(counts)]) & touched
    idx = np.where(keep)[0]
    return node_gids[idx], _compact(tris, idx, n)


def _find_bowties(n, tris):
    node_tris = [[] for _ in range(n)]
    for k, (a, b, c) in enumerate(tris):
        node_tris[a].append(k); node_tris[b].append(k); node_tris[c].append(k)
    bowties = []
    for node, tl in enumerate(node_tris):
        if len(tl) <= 1:
            continue
        wing = {}
        for k in tl:
            for w in tris[k]:
                if w != node:
                    wing.setdefault(w, []).append(k)
        parent = {k: k for k in tl}

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        for tl2 in wing.values():
            for j in range(1, len(tl2)):
                ra, rb = find(tl2[0]), find(tl2[j])
                if ra != rb:
                    parent[ra] = rb
        if len({find(k) for k in tl}) > 1:
            bowties.append(node)
    return bowties, node_tris


def mesh_from_mom6(grid, topo):
    """Triangulate the wet MOM6 T-cells of `grid`/`topo`. Returns a dict with lon/lat
    (deg), depth (m, positive), tris (0-based Nx3), and orig_j/orig_i/ny/nx (original
    T-grid indices, used for boundary classification)."""
    tlon, tlat = grid.tlon.values, grid.tlat.values
    tmask = topo.tmask.values.astype(bool)
    depth = topo.masked_depth.values
    ny, nx = tmask.shape
    gid = lambda j, i: j * nx + i

    tris = []
    for j in range(ny - 1):
        for i in range(nx - 1):
            sw, se, nw, ne = tmask[j, i], tmask[j, i + 1], tmask[j + 1, i], tmask[j + 1, i + 1]
            wet = (sw, se, nw, ne)
            n_wet = sum(wet)
            if n_wet < 3:
                continue
            g = (gid(j, i), gid(j, i + 1), gid(j + 1, i), gid(j + 1, i + 1))
            if n_wet == 4:
                d1 = _haversine_km(tlon[j, i], tlat[j, i], tlon[j + 1, i + 1], tlat[j + 1, i + 1])
                d2 = _haversine_km(tlon[j, i + 1], tlat[j, i + 1], tlon[j + 1, i], tlat[j + 1, i])
                if d1 <= d2:
                    tris += [(g[0], g[1], g[3]), (g[0], g[3], g[2])]
                else:
                    tris += [(g[0], g[1], g[2]), (g[1], g[3], g[2])]
            else:
                tris.append(tuple(gg for gg, w in zip(g, wet) if w))
    tris = np.array(tris, dtype=np.int64)

    node_gids = np.unique(tris.reshape(-1))
    tris = _compact(tris, node_gids, ny * nx)
    while True:
        node_gids, tris = _largest_component(node_gids, tris)
        bowties, node_tris = _find_bowties(len(node_gids), tris)
        if not bowties:
            break
        bad = {t for node in bowties for t in node_tris[node]}
        tris = tris[[k not in bad for k in range(len(tris))]]
        used = np.unique(tris.reshape(-1))
        tris = _compact(tris, used, len(node_gids))
        node_gids = node_gids[used]

    orig_j, orig_i = node_gids // nx, node_gids % nx
    return dict(lon=tlon[orig_j, orig_i], lat=tlat[orig_j, orig_i], depth=depth[orig_j, orig_i],
                tris=tris, orig_j=orig_j, orig_i=orig_i, ny=ny, nx=nx)


def read_msh(path):
    """Load a bring-your-own gmsh v2 mesh (see requirements above)."""
    lines = Path(path).read_text().splitlines()
    i, lon = 0, None
    tris = []
    while i < len(lines):
        line = lines[i].strip()
        if line == "$Nodes":
            n = int(lines[i + 1]); i += 2
            lon, lat, depth = np.zeros(n), np.zeros(n), np.zeros(n)
            for k in range(n):
                p = lines[i + k].split()
                idx = int(p[0]) - 1
                lon[idx], lat[idx], depth[idx] = float(p[1]), float(p[2]), float(p[3])
            i += n
        elif line == "$Elements":
            m = int(lines[i + 1]); i += 2
            for k in range(m):
                p = lines[i + k].split()
                if int(p[1]) == 2:  # type 2 = triangle: id type ntags [tags...] n1 n2 n3
                    ntag = int(p[2])
                    tris.append(tuple(int(x) - 1 for x in p[3 + ntag:3 + ntag + 3]))
            i += m
        else:
            i += 1
    if lon is None or not tris:
        raise ValueError(f"{path}: no $Nodes / triangle $Elements found")
    depth = np.asarray(depth)
    if np.any(depth <= 0):
        raise ValueError("read_msh: found depth <= 0; WW3 unstructured meshes need positive metres")
    return dict(lon=lon, lat=lat, depth=depth, tris=np.array(tris, dtype=np.int64))


if mesh_source == "mom6":
    mesh = mesh_from_mom6(grid, topo)
else:
    mesh = read_msh(mesh_source)
print(f"Mesh: {len(mesh['lon'])} nodes, {len(mesh['tris'])} triangles "
      f"(source={'mom6' if mesh_source == 'mom6' else mesh_source})")

## Step 4b: Boundary classification

**Open** = mesh-boundary nodes (on an edge used by exactly 1 triangle) that lie on one
of the case's open MOM6 edges (`boundaries`, passed to `configure_forcings` above).
Every other boundary node is **coast**. For a bring-your-own mesh there is no original
MOM6 grid index to match, so `classify_boundary_external` instead uses a distance
tolerance to the case's open-edge T-cell centres.

In [ ]:
def _boundary_nodes(tris, n):
    edge_count = {}
    for a, b, c in tris:
        for u, v in ((a, b), (b, c), (c, a)):
            key = (u, v) if u < v else (v, u)
            edge_count[key] = edge_count.get(key, 0) + 1
    nodes = sorted({u for (u, v), cnt in edge_count.items() if cnt == 1 for u in (u, v)})
    is_boundary = np.zeros(n, dtype=bool)
    is_boundary[nodes] = True
    return is_boundary


def classify_boundary_mom6(mesh, boundaries):
    n = len(mesh["lon"])
    is_boundary = _boundary_nodes(mesh["tris"], n)
    ny, nx, oj, oi = mesh["ny"], mesh["nx"], mesh["orig_j"], mesh["orig_i"]
    on_edge = np.zeros(n, dtype=bool)
    if "south" in boundaries: on_edge |= (oj == 0)
    if "north" in boundaries: on_edge |= (oj == ny - 1)
    if "west" in boundaries: on_edge |= (oi == 0)
    if "east" in boundaries: on_edge |= (oi == nx - 1)
    return is_boundary & on_edge, is_boundary


def classify_boundary_external(mesh, grid, boundaries, tol_km=None):
    n = len(mesh["lon"])
    is_boundary = _boundary_nodes(mesh["tris"], n)
    tlon, tlat = grid.tlon.values, grid.tlat.values
    edges_lon, edges_lat = [], []
    if "south" in boundaries: edges_lon.append(tlon[0, :]); edges_lat.append(tlat[0, :])
    if "north" in boundaries: edges_lon.append(tlon[-1, :]); edges_lat.append(tlat[-1, :])
    if "west" in boundaries: edges_lon.append(tlon[:, 0]); edges_lat.append(tlat[:, 0])
    if "east" in boundaries: edges_lon.append(tlon[:, -1]); edges_lat.append(tlat[:, -1])
    is_open = np.zeros(n, dtype=bool)
    if edges_lon:
        elon, elat = np.concatenate(edges_lon), np.concatenate(edges_lat)
        if tol_km is None:
            tol_km = 1.5 * _haversine_km(tlon[0, 0], tlat[0, 0], tlon[0, 1], tlat[0, 1])
        for k in np.where(is_boundary)[0]:
            if _haversine_km(mesh["lon"][k], mesh["lat"][k], elon, elat).min() <= tol_km:
                is_open[k] = True
    return is_open, is_boundary


if mesh_source == "mom6":
    is_open, is_boundary = classify_boundary_mom6(mesh, boundaries)
else:
    is_open, is_boundary = classify_boundary_external(mesh, grid, boundaries)
print(f"Boundary nodes: {int(is_boundary.sum())} "
      f"(open={int(is_open.sum())}, coast={int((is_boundary & ~is_open).sum())})")

## Step 4c: Plot the mesh


In [ ]:
import matplotlib.tri as mtri

plt.close()
fig, ax = plt.subplots(figsize=(6, 4))
ax.triplot(mtri.Triangulation(mesh["lon"], mesh["lat"], mesh["tris"]), linewidth=0.2, color="0.6")
coast = is_boundary & ~is_open
ax.plot(mesh["lon"][is_open], mesh["lat"][is_open], "r.", ms=3, label=f"open ({int(is_open.sum())})")
ax.plot(mesh["lon"][coast], mesh["lat"][coast], ".", color="saddlebrown", ms=3, label=f"coast ({int(coast.sum())})")
ax.set_xlabel("lon"); ax.set_ylabel("lat"); ax.set_aspect("equal")
ax.legend(loc="lower left", fontsize=8)
plt.show()

## Step 4d: Write WW3 inputs

Writes to `<inputdir>/wav_unst/`:

- the gmsh mesh file and the `UGOBCFILE` (open-boundary side-car);
- `ww3_grid.nml` (grid-preprocessor namelist: spectral/run/timestep values, unstructured
  grid record);
- `namelists.nml` — the `&OUTS`/`&LMPN`/`&SIC4`/`&MISC` physics blocks copied verbatim
  from **this case's own** `<inputdir>/wav/ww3_grid.inp` (so it tracks whatever CrocoDash
  version wrote it), plus an `&UNST` block (implicit by default; `scheme="explicit"`
  sets `EXPFSN=T` and the `IMP*` flags to `F`);
  copies of CrocoDash's boundary spectra files (`spec.list`, `ww3_bounc.nml`,
  `ww3.pointN_spec.nc`);
- the ESMF median-dual coupling mesh (below).

In [ ]:
import re
import shutil


def _extract_block(text, name):
    m = re.search(rf"&{name}\b.*?/", text, re.S)
    return m.group(0) if m else None


def _extract_spectrum_run_timesteps(text):
    """Pull the 3 positional records preceding the namelist blocks in a CrocoDash-
    written ww3_grid.inp: SPECTRUM (xfr freq1 nk nth thoff), RUN flags (fldry flcx
    flcy flcth flck flsou), TIMESTEPS (dtmax dtxy dtkth dtmin)."""
    nums = []
    for line in text.splitlines():
        s = line.strip()
        if not s or s.startswith("$") or s.startswith("'"):
            continue
        if re.match(r"^[\d.\-TFtf]", s):
            nums.append(s.split())
            if len(nums) == 3:
                break
    if len(nums) != 3:
        raise ValueError("Could not locate SPECTRUM/RUN/TIMESTEPS records in ww3_grid.inp")
    spec, run, ts = nums
    return dict(xfr=spec[0], freq1=spec[1], nk=spec[2], nth=spec[3], thoff=spec[4],
                flcx=run[1], flcy=run[2], flcth=run[3], flck=run[4], flsou=run[5],
                dtmax=ts[0], dtxy=ts[1], dtkth=ts[2], dtmin=ts[3])


def _cfl_dt(min_edge_km, freq1_hz, cpl_dt_s=1800.0):
    """Explicit-scheme timestep: min mesh edge length / deep-water group velocity at
    the lowest spectral frequency, rounded DOWN to a divisor of the coupling interval."""
    cg = 9.81 / (4 * np.pi * freq1_hz)  # deep-water group velocity, m/s
    dt_cfl = (min_edge_km * 1000.0) / cg
    for d in range(int(dt_cfl), 0, -1):
        if cpl_dt_s % d == 0:
            return float(d)
    return 1.0


def compute_timesteps(mesh, scheme, spec_run_ts, cpl_dt_s=1800.0):
    """Grid-preprocessor timesteps (s). Implicit: dtmax=dtxy=dtkth (no CFL limit
    needed). Explicit: dtxy=dtkth from the CFL constraint. dtmin is unchanged."""
    dtmax, dtmin = float(spec_run_ts["dtmax"]), float(spec_run_ts["dtmin"])
    if scheme == "implicit":
        return dict(dtmax=dtmax, dtxy=dtmax, dtkth=dtmax, dtmin=dtmin)
    edges = {(min(a, b), max(a, b)) for a, b, c in mesh["tris"] for a, b in ((a, b), (b, c), (c, a))}
    lon, lat = mesh["lon"], mesh["lat"]
    min_edge = min(_haversine_km(lon[u], lat[u], lon[v], lat[v]) for u, v in edges)
    dt = _cfl_dt(min_edge, float(spec_run_ts["freq1"]), cpl_dt_s)
    return dict(dtmax=dtmax, dtxy=dt, dtkth=dt, dtmin=dtmin)


def write_msh(path, lon, lat, depth, tris, boundary_nodes):
    with open(path, "w") as f:
        f.write("$MeshFormat\n2 0 8\n$EndMeshFormat\n$Nodes\n")
        f.write(f"{len(lon)}\n")
        for k in range(len(lon)):
            f.write(f"{k + 1} {lon[k]:.8f} {lat[k]:.8f} {depth[k]:.4f}\n")
        f.write(f"$EndNodes\n$Elements\n{len(tris) + len(boundary_nodes)}\n")
        eid = 0
        for a, b, c in tris:
            eid += 1
            f.write(f"{eid} 2 0 {a + 1} {b + 1} {c + 1}\n")
        for node in boundary_nodes:
            eid += 1
            f.write(f"{eid} 15 0 {node + 1}\n")
        f.write("$EndElements\n")


def write_ugobcfile(path, open_node_ids_1based):
    with open(path, "w") as f:
        f.write("$ Comments\n$ Open boundary points\n")
        for seq, node1 in enumerate(open_node_ids_1based, start=1):
            f.write(f"{seq} 0 {node1}\n")
        f.write("$ End of file\n")


def write_ww3_grid_nml(path, name, nml, min_depth, msh_name, obc_name):
    Path(path).write_text(f"""! Auto-generated by unstructured-ww3.ipynb (EXPERIMENTAL)
&SPECTRUM_NML
  SPECTRUM%XFR    = {nml['xfr']}
  SPECTRUM%FREQ1  = {nml['freq1']}
  SPECTRUM%NK     = {nml['nk']}
  SPECTRUM%NTH    = {nml['nth']}
  SPECTRUM%THOFF  = {nml['thoff']}
/

&RUN_NML
  RUN%FLCX   = {nml['flcx']}
  RUN%FLCY   = {nml['flcy']}
  RUN%FLCTH  = {nml['flcth']}
  RUN%FLCK   = {nml['flck']}
  RUN%FLSOU  = {nml['flsou']}
/

&TIMESTEPS_NML
  TIMESTEPS%DTMAX  = {nml['dtmax']}
  TIMESTEPS%DTXY   = {nml['dtxy']}
  TIMESTEPS%DTKTH  = {nml['dtkth']}
  TIMESTEPS%DTMIN  = {nml['dtmin']}
/

&GRID_NML
  GRID%NAME   = '{name}'
  GRID%NML    = './namelists.nml'
  GRID%TYPE   = 'UNST'
  GRID%COORD  = 'SPHE'
  GRID%CLOS   = 'F'
  GRID%ZLIM   = -0.1
  GRID%DMIN   = {min_depth:.2f}
/

&UNST_NML
  UNST%SF        = -1.0
  UNST%FILENAME  = './{msh_name}'
  UNST%IDLA      = 4
  UNST%FORMAT    = '(20f10.2)'
  UNST%UGOBCFILE = './{obc_name}'
/
""")


def write_namelists_nml(path, ww3_grid_inp_text, scheme, obc_name):
    physics = "\n".join(b for b in (_extract_block(ww3_grid_inp_text, n)
                                     for n in ("OUTS", "LMPN", "SIC4", "MISC")) if b)
    if scheme == "implicit":
        unst = f"""&UNST UGOBCAUTO = F,
UGOBCDEPTH = -10.,
UGOBCFILE = './{obc_name}',
EXPFSN = F,
IMPTOTAL = T,
IMPREFRACTION = T,
IMPFREQSHIFT = T,
IMPSOURCE = T,
JGS_USE_JACOBI = T,
JGS_BLOCK_GAUSS_SEIDEL = T,
JGS_TERMINATE_MAXITER = T,
JGS_MAXITER = 1000,
JGS_TERMINATE_NORM = F,
JGS_TERMINATE_DIFFERENCE = T,
JGS_DIFF_THR = 1.E-8,
JGS_PMIN = 3.0,
JGS_LIMITER = F,
JGS_NORM_THR = 1.E-20 /
"""
    else:
        unst = f"""&UNST UGOBCAUTO = F,
UGOBCDEPTH = -10.,
UGOBCFILE = './{obc_name}',
EXPFSN = T,
IMPTOTAL = F,
IMPREFRACTION = F,
IMPFREQSHIFT = F,
IMPSOURCE = F /
"""
    Path(path).write_text(
        "! Auto-generated by unstructured-ww3.ipynb (EXPERIMENTAL).\n"
        "! &OUTS/&LMPN/&SIC4/&MISC copied from this case's own <inputdir>/wav/ww3_grid.inp.\n\n"
        + physics + "\n" + unst + "END OF NAMELISTS\n")


def copy_boundary_spectra(inputdir):
    src, dst = Path(inputdir) / "wav", Path(inputdir) / "wav_unst"
    dst.mkdir(parents=True, exist_ok=True)
    for name in ("spec.list", "ww3_bounc.nml"):
        if (src / name).exists():
            shutil.copy(src / name, dst / name)
    for f in src.glob("ww3.point*_spec.nc"):
        shutil.copy(f, dst / f.name)

The ESMF median-dual coupling mesh has one element per WW3 node (`centerCoords` =
the node's own lon/lat). An **interior** node's dual polygon connects the centroids of
its incident triangles; a **boundary** node's is closed with the node's own position
plus the midpoints of its two boundary edges (a proper half-cell, not truncated at the
domain edge). Vertices are not deduplicated across elements — more verbose than a
shared-vertex-pool mesh, but an equally valid ESMF unstructured mesh.

In [ ]:
def build_dual_mesh(mesh, is_boundary):
    lon, lat, tris = mesh["lon"], mesh["lat"], mesh["tris"]
    n = len(lon)
    node_tris = [[] for _ in range(n)]
    for k, (a, b, c) in enumerate(tris):
        node_tris[a].append(k); node_tris[b].append(k); node_tris[c].append(k)
    cx, cy = lon[tris].mean(axis=1), lat[tris].mean(axis=1)  # triangle centroids

    edge_tris = {}
    for k, (a, b, c) in enumerate(tris):
        for u, v in ((a, b), (b, c), (c, a)):
            key = (u, v) if u < v else (v, u)
            edge_tris.setdefault(key, []).append(k)
    edge_mid = {e: ((lon[e[0]] + lon[e[1]]) / 2, (lat[e[0]] + lat[e[1]]) / 2)
                for e, tl in edge_tris.items() if len(tl) == 1}
    node_edges = [[] for _ in range(n)]
    for u, v in edge_mid:
        node_edges[u].append((u, v)); node_edges[v].append((u, v))

    pool_lon, pool_lat, conn_rows = [], [], []
    for node in range(n):
        pts = [(cx[k], cy[k]) for k in node_tris[node]]
        pts += [edge_mid[e] for e in node_edges[node]]
        if pts:
            ang = np.arctan2([p[1] - lat[node] for p in pts],
                              [(p[0] - lon[node]) * np.cos(np.radians(lat[node])) for p in pts])
            pts = [pts[k] for k in np.argsort(ang)]
        if is_boundary[node]:
            pts = [(lon[node], lat[node])] + pts
        start = len(pool_lon)
        pool_lon += [p[0] for p in pts]
        pool_lat += [p[1] for p in pts]
        conn_rows.append(list(range(start, start + len(pts))))

    max_conn = max(len(r) for r in conn_rows)
    elem_conn = -np.ones((n, max_conn), dtype=np.int32)
    n_conn = np.zeros(n, dtype=np.int32)
    for e, r in enumerate(conn_rows):
        elem_conn[e, :len(r)] = np.array(r) + 1  # 1-based
        n_conn[e] = len(r)
    return dict(node_lon=np.array(pool_lon), node_lat=np.array(pool_lat),
                elem_conn=elem_conn, n_conn=n_conn, center_lon=lon.copy(), center_lat=lat.copy())


def write_esmf_mesh(path, dual):
    import netCDF4 as nc
    n_node, n_elem = len(dual["node_lon"]), len(dual["center_lon"])
    with nc.Dataset(path, "w", format="NETCDF4") as ds:
        ds.gridType, ds.version = "unstructured mesh", "0.9"
        ds.createDimension("nodeCount", n_node)
        ds.createDimension("elementCount", n_elem)
        ds.createDimension("maxNodePElement", dual["elem_conn"].shape[1])
        ds.createDimension("coordDim", 2)
        v = ds.createVariable("nodeCoords", "f8", ("nodeCount", "coordDim")); v.units = "degrees"
        v[:, 0], v[:, 1] = dual["node_lon"], dual["node_lat"]
        ec = ds.createVariable("elementConn", "i4", ("elementCount", "maxNodePElement"), fill_value=-1)
        ec.long_name = "Node indices that define the element connectivity"
        ec[:, :] = dual["elem_conn"]
        nec = ds.createVariable("numElementConn", "i4", ("elementCount",))
        nec.long_name = "Number of nodes per element"
        nec[:] = dual["n_conn"]
        v = ds.createVariable("centerCoords", "f8", ("elementCount", "coordDim")); v.units = "degrees"
        v[:, 0], v[:, 1] = dual["center_lon"], dual["center_lat"]
        ds.createVariable("elementMask", "i4", ("elementCount",))[:] = np.ones(n_elem, dtype=np.int32)


def verify_esmf_mesh(path):
    try:
        import esmpy as ESMF
    except ImportError:
        try:
            import ESMF
        except ImportError:
            print("esmpy not available in this env; skipping load-verification.")
            return
    ESMF.Mesh(filename=str(path), filetype=ESMF.FileFormat.ESMFMESH)
    print(f"esmpy loaded {path} OK")


def write_wav_unst(inputdir, topo, mesh, is_open, is_boundary, name, scheme):
    wav_unst = Path(inputdir) / "wav_unst"
    wav_unst.mkdir(parents=True, exist_ok=True)
    msh_path, obc_path = wav_unst / f"{name}.msh", wav_unst / f"{name}_obc.mshb"
    write_msh(msh_path, mesh["lon"], mesh["lat"], mesh["depth"], mesh["tris"], np.where(is_boundary)[0])
    write_ugobcfile(obc_path, np.where(is_open)[0] + 1)

    grid_inp_text = (Path(inputdir) / "wav" / "ww3_grid.inp").read_text()
    spec_run_ts = _extract_spectrum_run_timesteps(grid_inp_text)
    ts = compute_timesteps(mesh, scheme, spec_run_ts)
    write_ww3_grid_nml(wav_unst / "ww3_grid.nml", name, {**spec_run_ts, **ts},
                        topo.min_depth, msh_path.name, obc_path.name)
    write_namelists_nml(wav_unst / "namelists.nml", grid_inp_text, scheme, obc_path.name)
    copy_boundary_spectra(inputdir)

    dual = build_dual_mesh(mesh, is_boundary)
    dual_path = wav_unst / f"{name}_ESMFmesh.nc"
    write_esmf_mesh(dual_path, dual)
    verify_esmf_mesh(dual_path)
    return dict(spec_run_ts=spec_run_ts, ts=ts, dual_path=dual_path, wav_unst=wav_unst)


wav_result = write_wav_unst(inputdir, topo, mesh, is_open, is_boundary, wave_grid_name, scheme)
print(f"Wrote WW3 unstructured inputs to {wav_result['wav_unst']} "
      f"(timesteps: {wav_result['ts']})")

## Step 4e: Configure the case

Applies the `xmlchange`s (`--non-local` when not running on derecho), edits
`user_nl_ww3` (replacing any existing `dtmax`/`dtcfl`/`dtcfli` lines rather than
duplicating them), sets the coupler history cadence, and moves stale structured WW3
files (`ww3_grid.inp`, `*_x/_y/_bottom/_mapsta.inp`) out of
`RUNDIR/ww3_moddef_create` — CIME's buildnml only copies `WW3_GRID_INP_DIR` files that
are not already there, and CrocoDash pre-copies the structured ones at case creation.
Refuses (unless `force=True`) if the case has already been built, since any of these
XML changes after a first build requires `./case.build --clean-all`.

In [ ]:
import socket
import subprocess


def _xmlchange(caseroot, var, val, non_local):
    cmd = ["./xmlchange", f"{var}={val}"] + (["--non-local"] if non_local else [])
    r = subprocess.run(cmd, cwd=caseroot, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stdout}\n{r.stderr}")


def _is_on_derecho():
    return "derecho" in socket.gethostname().lower()


def _xmlquery(caseroot, var):
    cmd = ["./xmlquery", var, "--value"] + ([] if _is_on_derecho() else ["--non-local"])
    r = subprocess.run(cmd, cwd=caseroot, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stdout}\n{r.stderr}")
    return r.stdout.strip()


def _set_user_nl(caseroot, updates):
    path = Path(caseroot) / "user_nl_ww3"
    lines = path.read_text().splitlines() if path.exists() else []
    remaining = dict(updates)
    out = []
    for line in lines:
        m = re.match(r"\s*([A-Za-z_]+)\s*=", line)
        if m and m.group(1) in remaining:
            out.append(f"{m.group(1)} = {remaining.pop(m.group(1))}")
        else:
            out.append(line)
    out += [f"{k} = {v}" for k, v in remaining.items()]
    path.write_text("\n".join(out) + "\n")


def _clean_stale_structured(rundir):
    moddef = Path(rundir) / "ww3_moddef_create"
    if not moddef.exists():
        return []
    backup, moved = moddef / "curv_bak", []
    for pat in ("ww3_grid.inp", "*_x.inp", "*_y.inp", "*_bottom.inp", "*_mapsta.inp"):
        for f in moddef.glob(pat):
            backup.mkdir(exist_ok=True)
            shutil.move(str(f), str(backup / f.name))
            moved.append(f.name)
    return moved


def configure_unstructured_ww3(case, mesh, is_open, name, wav_result, force=False):
    caseroot = str(case.caseroot)
    non_local = not _is_on_derecho()
    if _xmlquery(caseroot, "BUILD_COMPLETE").upper() == "TRUE" and not force:
        raise RuntimeError(
            "This case has already been built (BUILD_COMPLETE=TRUE). Changing WW3_UNSTRUCT/"
            "WAV_GRID/WAV_NX/WW3_GRID_INP_DIR after the first build requires a full "
            "./case.build --clean-all. Pass force=True to proceed anyway (you still need to "
            "run --clean-all yourself before the next build)."
        )
    check_cesmroot_patches(case.cesmroot)
    n_nodes = len(mesh["lon"])
    for var, val in [("WW3_UNSTRUCT", "TRUE"), ("WAV_GRID", name), ("WAV_NX", n_nodes),
                      ("WAV_NY", 1), ("WW3_GRID_INP_DIR", str(wav_result["wav_unst"])),
                      ("WAV_DOMAIN_MESH", str(wav_result["dual_path"]))]:
        _xmlchange(caseroot, var, val, non_local)
    ts = wav_result["ts"]
    _set_user_nl(caseroot, {"dtmax": ts["dtmax"], "dtcfl": ts["dtxy"], "dtcfli": ts["dtkth"],
                            "history_option": "'nhours'", "history_n": 6})
    _xmlchange(caseroot, "HIST_OPTION", "nhours", non_local)
    _xmlchange(caseroot, "HIST_N", 6, non_local)
    moved = _clean_stale_structured(_xmlquery(caseroot, "RUNDIR"))
    print(f"Configured {caseroot}: WAV_GRID={name} WAV_NX={n_nodes} scheme timesteps={ts}. "
          f"Moved stale structured files to curv_bak: {moved or 'none found'}.")


configure_unstructured_ww3(case, mesh, is_open, wave_grid_name, wav_result)

# Section 5: Build and run the case

Build and submit on **Derecho** (not Casper), same as the base tutorial:
```
ssh derecho
cd <casedir>/bering_unst_ww3.001
qcmd -A <PROJECT> -- ./case.build
./case.submit
```

For a quick test, a small PE layout in the `develop` queue avoids a long queue wait
(`develop` allows <=63 tasks). Set `JOB_QUEUE` *before* `case.setup --reset`, not just
at submit time -- `case.setup` checks `NTASKS` against whatever `JOB_QUEUE` is
already set to, and fails with "ERROR: No queues found" if the two disagree (e.g.
48 tasks with the still-default `JOB_QUEUE=main`, which needs >=64):
```
./xmlchange JOB_QUEUE=develop
./xmlchange NTASKS=48,ROOTPE=0
./xmlchange NTASKS_ESP=1
./xmlchange MEM_PER_TASK=3
./case.setup --reset
qcmd -A <PROJECT> -q develop -- ./case.build
./case.submit
```
(`case.submit` takes no `VAR=VALUE` arguments -- unlike `xmlchange`, a bare
`JOB_QUEUE=develop` here is parsed as the `caseroot` positional argument.)

What to check:
- `RUNDIR/ww3_moddef_create/mod_def.ww3.log`: "Number of input b. points" should equal
  the open-node count printed in Step 4b.
- `RUNDIR/ww3_moddef_create/nest.ww3.log` / `nest.ww3` should exist (boundary spectra).
- `CaseStatus`: look for `case.run success` (or `case.run error`).

# Output: plot significant wave height (Hs)

Looks for a WW3 history file in `RUNDIR` first, then the short-term archive, and plots
`HS` with `tripcolor` using the history file's own triangle connectivity (`nconn`,
1-based). Also prints `HS` on the open-boundary nodes (`mapsta == 2`).

In [ ]:
import xarray as xr


def find_history_file(case, pattern="*.ww3.hi.*.nc"):
    caseroot = str(case.caseroot)
    candidates = sorted(Path(_xmlquery(caseroot, "RUNDIR")).glob(pattern))
    if _xmlquery(caseroot, "DOUT_S").upper() == "TRUE":
        candidates += sorted((Path(_xmlquery(caseroot, "DOUT_S_ROOT")) / "wav" / "hist").glob(pattern))
    if not candidates:
        raise FileNotFoundError(f"No WW3 history file matching {pattern} in RUNDIR or the archive.")
    return candidates[-1]


def plot_hs(history_path):
    ds = xr.open_dataset(history_path)
    hs = ds["HS"].isel(time=-1).values.reshape(-1) if "time" in ds["HS"].dims else ds["HS"].values.reshape(-1)
    nconn = ds["nconn"].values.astype(int) - 1  # 1-based -> 0-based
    lon, lat = ds["lon"].values.reshape(-1), ds["lat"].values.reshape(-1)
    plt.close()
    fig, ax = plt.subplots(figsize=(6, 4))
    tp = ax.tripcolor(mtri.Triangulation(lon, lat, nconn), hs, shading="flat")
    plt.colorbar(tp, ax=ax, label="Hs (m)")
    ax.set_aspect("equal")
    plt.show()
    mapsta = ds["mapsta"].isel(time=-1).values.reshape(-1) if "time" in ds["mapsta"].dims else ds["mapsta"].values.reshape(-1)
    open_hs = hs[mapsta == 2]
    print(f"Hs on {len(open_hs)} open-boundary nodes: "
          f"min={open_hs.min():.3f} mean={open_hs.mean():.3f} max={open_hs.max():.3f} m")


hist_path = find_history_file(case)
plot_hs(hist_path)

# Known limitations

- **Implicit solver**: hits `JGS_MAXITER` every step (results still matched a
  structured twin at rms 0.02 m / corr 1.000 on the validated Bering test case) — not
  a bug to chase right now.
- **No pole**: neither `mesh_from_mom6` nor `read_msh` handles a mesh containing a
  pole.
- **CICE coupling untested** on an unstructured wave mesh.
- **Ice physics** depends entirely on CrocoDash's own `&SIC4` settings (`IC4METHOD`),
  copied here, not re-tuned.
- **Performance**: this is a prototype pipeline, not a performance-tuned one — PE
  layout, PDLIB decomposition quality, etc. are all out of scope here.
- This whole notebook is a **prototype**: rough edges are expected. Please report
  problems (mesh gotchas, build/run failures, namelist surprises) to the
  CROCODILE-CESM team.